# Tahap 2: Ekstraksi Basis Aturan Strategis dengan Association Rule Mining (ARM)
**Penelitian:** HOK Virtual Coach: Model Rekomendasi Draft Pick pada Game Honor of Kings Berbasis Association Rule Mining (ARM) dan Long Short-Term Memory (LSTM)  
**Oleh:** Adiyatma Admaja - NIM 082111633061 (S-1 Sistem Informasi, Universitas Airlangga)  
**Referensi Naskah:** BAB II Subbab 2.2, BAB III Subbab 3.2.1 (Pembuatan Model ARM), & BAB IV (Hasil Ekstraksi Aturan Asosiasi)  
**Urutan Pipeline:** Tahap 2 dari 5 (Menggunakan output Notebook 1; menghasilkan 5 Basis Aturan untuk Notebook 5)

## 1. Tujuan Notebook
Notebook ini bertujuan mengekstraksi pola ko-okurensi pemilihan dan pelarangan *hero* dari data historis menggunakan algoritma Apriori, kemudian menyaring dan mengurutkannya berdasarkan *domain knowledge* mekanisme *draft pick* ke dalam lima Basis Aturan (*Rule Base*) strategis.

## 2. Spesifikasi Input
* `arm_picks_cleaned.csv`: Data historis tindakan pemilihan *hero* (5.580 baris *pick* dari 558 pertandingan bersih).
* `arm_bans_cleaned.csv`: Data historis tindakan pelarangan *hero* (4.464 baris *ban* dari 558 pertandingan bersih).

## 3. Alur Proses & Pembentukan 5 Basis Aturan (Tabel 3.4 & Tabel 3.5)
Menggunakan pustaka `mlxtend` (`TransactionEncoder`, `apriori`, dan `association_rules`) dengan parameter ambang batas `MIN_SUPPORT = 0.0001` (0,01%) untuk menambang lima basis aturan spesifik:
1. **Aturan 1 – Synergy Base / Basis Kawan (`rules_allies`):** Mengekstraksi kombinasi *hero* dalam tim pemenang (`Win_ -> Win_`) dengan batas maksimum $\le 5$ item dan mengurutkannya berdasarkan metrik **Win Rate** (Persamaan 3.1) serta `support` (**79.402 aturan**).
2. **Aturan 2 – Counter Base / Basis Lawan (`rules_counters`):** Mengekstraksi hubungan kontra 1 lawan 1 antara *hero* tim yang kalah dan *hero* tim pemenang (`Lose_Enemy -> Win_Counter`), diurutkan berdasarkan `confidence` dan **Counter Coefficient** (Persamaan 3.2) (**3.691 aturan**).
3. **Aturan 3 – Meta Bans (`rules_meta_ban`):** Mengekstraksi pola ko-okurensi *hero* yang dilarang oleh tim pemenang untuk mengidentifikasi prioritas pelarangan di fase awal (**19.706 aturan**).
4. **Aturan 4 – Combo Breaker Bans (`rules_combo_breaker`):** Mengekstraksi aturan sekuensial dari 3 *pick* awal lawan menuju 2 *ban* fase kedua tim sendiri (`EnemyPick_ -> OurBan_`) guna memutus sinergi komposisi lawan (**16.160 aturan**).
5. **Aturan 5 – Anti-Counter Bans (`rules_anti_counter`):** Mengekstraksi aturan sekuensial dari 3 *pick* awal tim sendiri menuju 2 *ban* fase kedua tim sendiri (`OurPick_ -> OurBan_`) guna mengamankan *hero* inti dari ancaman kontra lawan (**16.736 aturan**).

## 4. Artefak Output yang Dihasilkan
* Tabel distribusi frekuensi aturan asosiasi berdasarkan rentang *support* (total keseluruhan **135.695 aturan**).
* Lima berkas CSV basis aturan pada direktori `Extracted-ARM-Rules/`: `rules_allies.csv`, `rules_counters.csv`, `rules_meta_ban.csv`, `rules_combo_breaker.csv`, dan `rules_anti_counter.csv` untuk diintegrasikan pada Notebook 5.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import warnings

# Association Rule Mining

#### 1. PERSIAPAN DATA (PREPROCESSING)

In [2]:
# ==========================================
# 1. PERSIAPAN DATA (PREPROCESSING)
# ==========================================

# 1. Memuat Data yang sudah dibersihkan
df_picks = pd.read_csv('../Data/arm_picks_cleaned.csv')
df_bans = pd.read_csv('../Data/arm_bans_cleaned.csv')

# Gabungkan untuk kebutuhan sekuensial (Combo Breaker & Anti Counter)
df_full = pd.concat([df_picks, df_bans]).sort_values(by=['Match_Sequence', 'Draft_Order'])

MIN_SUPPORT = 0.0001

# 2. Fungsi Bantuan: Visualisasi Distribusi Support
def get_distribution(support_series, title):
    bins = [0.0001, 0.0002, 0.0005, 0.001, 0.01, 0.1, 1.0]
    labels = [
        '0.01% < s <= 0.02%',
        '0.02% < s <= 0.05%',
        '0.05% < s <= 0.10%',
        '0.10% < s <= 1.00%',
        '1.00% < s <= 10.0%',
        '> 10.0%'
    ]

    dist_categories = pd.cut(support_series, bins=bins, labels=labels, right=True)
    dist_counts = dist_categories.value_counts().reindex(labels[::-1]).fillna(0).astype(int)
    
    df_dist = pd.DataFrame({
        'Support Range': dist_counts.index,
        'Total Rules': dist_counts.values
    })

    total_rules = dist_counts.sum()
    total_row = pd.DataFrame({
        'Support Range': ['TOTAL'],
        'Total Rules': [total_rules]
    })

    df_dist = pd.concat([df_dist, total_row], ignore_index=True)

    print(f"\n--- DISTRIBUSI: {title} ---")
    print(df_dist.to_string(index=False))

    return df_dist

#### 2. EKSTRAKSI ATURAN ASOSIASI DENGAN APRIORI

In [3]:
# ==========================================
# ATURAN 1 (ALLIES / BASIS KAWAN)
# ==========================================
basket_allies = []

# Ekstrak data Win/Lose per tim
for (match_id, team_name), group in df_picks.groupby(['Match_ID', 'Team_Name']):
    is_win = group['Is_Win'].iloc[0]
    heroes = group['Hero_Name'].tolist()
    if is_win: basket_allies.append([f"Win_{h}" for h in heroes])
    else: basket_allies.append([f"Lose_{h}" for h in heroes])

te_allies = TransactionEncoder()
df_basket_allies = pd.DataFrame(te_allies.fit(basket_allies).transform(basket_allies), columns=te_allies.columns_)

freq_allies = apriori(df_basket_allies, min_support=MIN_SUPPORT, use_colnames=True)
rules_ap_allies = association_rules(freq_allies, metric="support", min_threshold=MIN_SUPPORT)
rules_ap_allies = rules_ap_allies[(rules_ap_allies['antecedents'].apply(len) + rules_ap_allies['consequents'].apply(len)) <= 5]

# Filter aturan (Win -> Win)
def is_valid_allies(ant, con):
    return all(str(i).startswith('Win_') for i in ant) and all(str(i).startswith('Win_') for i in con)

rules_allies = rules_ap_allies[rules_ap_allies.apply(lambda row: is_valid_allies(row['antecedents'], row['consequents']), axis=1)].copy()

# Strict Win Rate Kalkulasi
support_dict_allies = pd.Series(freq_allies.support.values, index=freq_allies.itemsets).to_dict()

def calc_win_rate(row):
    full_lose = frozenset([i.replace('Win_', 'Lose_') for i in row['antecedents'].union(row['consequents'])])
    s_win, s_lose = row['support'], support_dict_allies.get(full_lose, 0.0)
    return s_win / (s_win + s_lose) if (s_win + s_lose) > 0 else 0.0

rules_allies['win_rate'] = rules_allies.apply(calc_win_rate, axis=1)
rules_allies = rules_allies.sort_values(by=['win_rate', 'support'], ascending=[False, False])

# Format Output
rules_allies['Team_Heroes'] = rules_allies['antecedents'].apply(lambda x: [i.replace('Win_', '') for i in x])
rules_allies['Rec_Ally'] = rules_allies['consequents'].apply(lambda x: [i.replace('Win_', '') for i in x])

print("TOP 5 ATURAN 1 (ALLIES)")
print(rules_allies[['Team_Heroes', 'Rec_Ally', 'support', 'win_rate']].head(5).to_string(index=False))
get_distribution(rules_allies['support'], "ATURAN 1 (ALLIES)")

TOP 5 ATURAN 1 (ALLIES)
Team_Heroes    Rec_Ally  support  win_rate
   [Mayene]      [Luna] 0.007168       1.0
     [Luna]    [Mayene] 0.007168       1.0
   [Chicha]      [Haya] 0.006272       1.0
     [Haya]    [Chicha] 0.006272       1.0
     [Haya] [Zhang_Fei] 0.006272       1.0

--- DISTRIBUSI: ATURAN 1 (ALLIES) ---
     Support Range  Total Rules
           > 10.0%            0
1.00% < s <= 10.0%          188
0.10% < s <= 1.00%        10480
0.05% < s <= 0.10%        68734
0.02% < s <= 0.05%            0
0.01% < s <= 0.02%            0
             TOTAL        79402


,Support Range,Total Rules
0,> 10.0%,0
1,1.00% < s <= 10.0%,188
2,0.10% < s <= 1.00%,10480
3,0.05% < s <= 0.10%,68734
4,0.02% < s <= 0.05%,0
5,0.01% < s <= 0.02%,0
6,TOTAL,79402


In [4]:
# ==========================================
# ATURAN 2 (COUNTERS / BASIS LAWAN)
# ==========================================
basket_counters = []

# Ekstrak data Lose(Musuh) dan Win(Kita) per match
for match_id, match_data in df_picks.groupby('Match_ID'):
    heroes_menang = match_data[match_data['Is_Win'] == True]['Hero_Name'].tolist()
    heroes_kalah = match_data[match_data['Is_Win'] == False]['Hero_Name'].tolist()
    basket_counters.append([f"Lose_{h}" for h in heroes_kalah] + [f"Win_{h}" for h in heroes_menang])

te_counter = TransactionEncoder()
df_basket_counter = pd.DataFrame(te_counter.fit(basket_counters).transform(basket_counters), columns=te_counter.columns_)

freq_counters = apriori(df_basket_counter, min_support=MIN_SUPPORT, use_colnames=True)
rules_ap_counters = association_rules(freq_counters, metric="support", min_threshold=MIN_SUPPORT)

# Filter (1 Musuh -> 1 Counter)
def is_valid_counter(ant, con):
    if len(ant) != 1 or len(con) != 1: return False
    return all(str(i).startswith('Lose_') for i in ant) and all(str(i).startswith('Win_') for i in con)

rules_counters = rules_ap_counters[rules_ap_counters.apply(lambda r: is_valid_counter(r['antecedents'], r['consequents']), axis=1)].copy()

# Kalkulasi Counter Coefficient
support_dict_counters = pd.Series(freq_counters.support.values, index=freq_counters.itemsets).to_dict()

def calc_counter_coef(row):
    hero_e = list(row['antecedents'])[0].replace('Lose_', '')
    hero_r = list(row['consequents'])[0].replace('Win_', '')
    s_e_to_r = row['support']
    s_r_to_e = support_dict_counters.get(frozenset({f"Lose_{hero_r}", f"Win_{hero_e}"}), 0.0)
    return s_e_to_r / (s_e_to_r + s_r_to_e) if (s_e_to_r + s_r_to_e) > 0 else 0.0

rules_counters['counter_coef'] = rules_counters.apply(calc_counter_coef, axis=1)
rules_counters = rules_counters.sort_values(by=['confidence', 'counter_coef'], ascending=[False, False])

# Format Output
rules_counters['Enemy'] = rules_counters['antecedents'].apply(lambda x: list(x)[0].replace('Lose_', ''))
rules_counters['Rec_Counter'] = rules_counters['consequents'].apply(lambda x: list(x)[0].replace('Win_', ''))

print("TOP 5 ATURAN 2 (COUNTERS)")
print(rules_counters[['Enemy', 'Rec_Counter', 'support', 'confidence', 'counter_coef']].head(5).to_string(index=False))
get_distribution(rules_counters['support'], "ATURAN 2 (COUNTERS)")

TOP 5 ATURAN 2 (COUNTERS)
  Enemy  Rec_Counter  support  confidence  counter_coef
Alessio       Dharma 0.001792         1.0           1.0
Alessio Mai_Shiranui 0.001792         1.0           1.0
Alessio   Marco_Polo 0.001792         1.0           1.0
Alessio      Musashi 0.001792         1.0           1.0
Alessio    Zhang_Fei 0.001792         1.0           1.0

--- DISTRIBUSI: ATURAN 2 (COUNTERS) ---
     Support Range  Total Rules
           > 10.0%            0
1.00% < s <= 10.0%          778
0.10% < s <= 1.00%         2913
0.05% < s <= 0.10%            0
0.02% < s <= 0.05%            0
0.01% < s <= 0.02%            0
             TOTAL         3691


,Support Range,Total Rules
0,> 10.0%,0
1,1.00% < s <= 10.0%,778
2,0.10% < s <= 1.00%,2913
3,0.05% < s <= 0.10%,0
4,0.02% < s <= 0.05%,0
5,0.01% < s <= 0.02%,0
6,TOTAL,3691


In [5]:
# ==========================================
# CELL 4: ATURAN 3 (META / EFFECTIVE BANS)
# ==========================================
# Ambil daftar ban dari tim yang berhasil menang
basket_meta_ban = df_bans[df_bans['Is_Win'] == True].groupby(['Match_ID', 'Team_Name'])['Hero_Name'].apply(list).tolist()

te_meta = TransactionEncoder()
df_basket_meta = pd.DataFrame(te_meta.fit(basket_meta_ban).transform(basket_meta_ban), columns=te_meta.columns_)

freq_meta = apriori(df_basket_meta, min_support=MIN_SUPPORT, use_colnames=True)
rules_meta_ban = association_rules(freq_meta, metric="support", min_threshold=MIN_SUPPORT)
rules_meta_ban = rules_meta_ban.sort_values(by='support', ascending=False)

# Format Output (Apriori biasa)
rules_meta_ban['Target_Ban'] = rules_meta_ban['consequents'].apply(lambda x: list(x))

print("TOP 5 ATURAN 3 (META BANS)")
print(rules_meta_ban[['antecedents', 'Target_Ban', 'support', 'confidence']].head(5).to_string(index=False))
get_distribution(rules_meta_ban['support'], "ATURAN 3 (META BANS)")

TOP 5 ATURAN 3 (META BANS)
   antecedents     Target_Ban  support  confidence
        (Arli) [Mai_Shiranui] 0.069767    0.339130
(Mai_Shiranui)         [Arli] 0.069767    0.254902
(Mai_Shiranui)       [Dharma] 0.060823    0.222222
      (Dharma) [Mai_Shiranui] 0.060823    0.250000
        (Luna)          [Pei] 0.053667    0.201342

--- DISTRIBUSI: ATURAN 3 (META BANS) ---
     Support Range  Total Rules
           > 10.0%            0
1.00% < s <= 10.0%          374
0.10% < s <= 1.00%        19332
0.05% < s <= 0.10%            0
0.02% < s <= 0.05%            0
0.01% < s <= 0.02%            0
             TOTAL        19706


,Support Range,Total Rules
0,> 10.0%,0
1,1.00% < s <= 10.0%,374
2,0.10% < s <= 1.00%,19332
3,0.05% < s <= 0.10%,0
4,0.02% < s <= 0.05%,0
5,0.01% < s <= 0.02%,0
6,TOTAL,19706


In [6]:
# ---------------------------------------------------------
# PERSIAPAN KERANJANG SEKUENSIAL (Combo Breaker & Anti-Counter)
# ---------------------------------------------------------
basket_combo_breaker = []
basket_anti_counter = []

for match_id, match_data in df_full.groupby('Match_ID'):
    match_data = match_data.sort_values(by='Draft_Order')
    
    for team in ['Blue', 'Red']:
        enemy_team = 'Red' if team == 'Blue' else 'Blue'
        
        # Ekstrak Pick Fase 1
        our_picks = match_data[(match_data['Is_Ban']==False) & (match_data['Team_Side']==team)].head(3)['Hero_Name'].tolist()
        enemy_picks = match_data[(match_data['Is_Ban']==False) & (match_data['Team_Side']==enemy_team)].head(3)['Hero_Name'].tolist()
        
        # Ekstrak Ban Fase 2 (2 ban terakhir per tim)
        our_phase2_bans = match_data[(match_data['Is_Ban']==True) & (match_data['Team_Side']==team)].tail(2)['Hero_Name'].tolist()
        
        if our_phase2_bans:
            # Combo Breaker format (Pick Musuh -> Ban Kita)
            basket_combo_breaker.append([f"EnemyPick_{h}" for h in enemy_picks] + [f"OurBan_{h}" for h in our_phase2_bans])
            # Anti-Counter format (Pick Kita -> Ban Kita)
            basket_anti_counter.append([f"OurPick_{h}" for h in our_picks] + [f"OurBan_{h}" for h in our_phase2_bans])

te_seq = TransactionEncoder()

# ---------------------------------------------------------
# ATURAN 4: COMBO BREAKER BAN
# ---------------------------------------------------------
df_basket_combo_breaker = pd.DataFrame(te_seq.fit(basket_combo_breaker).transform(basket_combo_breaker), columns=te_seq.columns_)
freq_combo_breaker = apriori(df_basket_combo_breaker, min_support=MIN_SUPPORT, use_colnames=True)
rules_combo_breaker = association_rules(freq_combo_breaker, metric="support", min_threshold=MIN_SUPPORT)

# Filter: Antecedents = EnemyPick, Consequents = OurBan
rules_combo_breaker = rules_combo_breaker[
    rules_combo_breaker['antecedents'].apply(lambda x: all(str(i).startswith('EnemyPick_') for i in x)) &
    rules_combo_breaker['consequents'].apply(lambda x: all(str(i).startswith('OurBan_') for i in x))
]


print("\n--- TOP 5 ATURAN COMBO BREAKER (PICK MUSUH -> BAN KITA) ---")
print(rules_combo_breaker[['antecedents', 'consequents', 'support', 'confidence']].head(5))
get_distribution(rules_combo_breaker['support'], "ATURAN 4 (COMBO BREAKER)")


--- TOP 5 ATURAN COMBO BREAKER (PICK MUSUH -> BAN KITA) ---
          antecedents      consequents   support  confidence
49  (EnemyPick_Agudo)  (OurBan_Allain)  0.001792    0.035714
51  (EnemyPick_Agudo)  (OurBan_Angela)  0.001792    0.035714
53  (EnemyPick_Agudo)    (OurBan_Arli)  0.002688    0.053571
55  (EnemyPick_Agudo)     (OurBan_Ata)  0.001792    0.035714
57  (EnemyPick_Agudo)   (OurBan_Biron)  0.000896    0.017857

--- DISTRIBUSI: ATURAN 4 (COMBO BREAKER) ---
     Support Range  Total Rules
           > 10.0%            0
1.00% < s <= 10.0%           98
0.10% < s <= 1.00%         2828
0.05% < s <= 0.10%        13234
0.02% < s <= 0.05%            0
0.01% < s <= 0.02%            0
             TOTAL        16160


,Support Range,Total Rules
0,> 10.0%,0
1,1.00% < s <= 10.0%,98
2,0.10% < s <= 1.00%,2828
3,0.05% < s <= 0.10%,13234
4,0.02% < s <= 0.05%,0
5,0.01% < s <= 0.02%,0
6,TOTAL,16160


In [7]:
# ---------------------------------------------------------
# ATURAN 5: ANTI-COUNTER BAN
# ---------------------------------------------------------
df_basket_anti_counter = pd.DataFrame(te_seq.fit(basket_anti_counter).transform(basket_anti_counter), columns=te_seq.columns_)
freq_anti_counter = apriori(df_basket_anti_counter, min_support=MIN_SUPPORT, use_colnames=True)
rules_anti_counter = association_rules(freq_anti_counter, metric="support", min_threshold=MIN_SUPPORT)

# Filter: Antecedents = OurPick, Consequents = OurBan
rules_anti_counter = rules_anti_counter[
    rules_anti_counter['antecedents'].apply(lambda x: all(str(i).startswith('OurPick_') for i in x)) &
    rules_anti_counter['consequents'].apply(lambda x: all(str(i).startswith('OurBan_') for i in x))
]

print("\n--- TOP 5 ATURAN ANTI-COUNTER (PICK KITA -> BAN KITA) ---")
print(rules_anti_counter[['antecedents', 'consequents', 'support', 'confidence']].head(5))
get_distribution(rules_anti_counter['support'], "ATURAN 5 (ANTI-COUNTER)")


--- TOP 5 ATURAN ANTI-COUNTER (PICK KITA -> BAN KITA) ---
         antecedents     consequents   support  confidence
53  (OurPick_Allain)  (OurBan_Agudo)  0.001792    0.100000
54  (OurPick_Ao_Yin)  (OurBan_Agudo)  0.009857    0.050691
56    (OurPick_Arli)  (OurBan_Agudo)  0.010753    0.076433
59  (OurPick_Augran)  (OurBan_Agudo)  0.011649    0.138298
61   (OurPick_Biron)  (OurBan_Agudo)  0.003584    0.030534

--- DISTRIBUSI: ATURAN 5 (ANTI-COUNTER) ---
     Support Range  Total Rules
           > 10.0%            0
1.00% < s <= 10.0%           71
0.10% < s <= 1.00%         2700
0.05% < s <= 0.10%        13965
0.02% < s <= 0.05%            0
0.01% < s <= 0.02%            0
             TOTAL        16736


,Support Range,Total Rules
0,> 10.0%,0
1,1.00% < s <= 10.0%,71
2,0.10% < s <= 1.00%,2700
3,0.05% < s <= 0.10%,13965
4,0.02% < s <= 0.05%,0
5,0.01% < s <= 0.02%,0
6,TOTAL,16736


In [8]:
# ==========================================
# RINGKASAN TOTAL 5 ATURAN
# ==========================================
total_allies = len(rules_allies)
total_counters = len(rules_counters)
total_meta_ban = len(rules_meta_ban)
total_combo_breaker = len(rules_combo_breaker)
total_anti_counter = len(rules_anti_counter)
total_keseluruhan = total_allies + total_counters + total_meta_ban + total_combo_breaker + total_anti_counter

print("==========================================")
print("     📊 RINGKASAN DATASET & ATURAN 📊     ")
print("==========================================")
print(f"Total Pertandingan : {df_full['Match_ID'].nunique()} Match")
print("------------------------------------------")
print(f"1. Aturan Basis Kawan (Allies)     : {total_allies} Rules")
print(f"2. Aturan Basis Lawan (Counters)   : {total_counters} Rules")
print(f"3. Aturan Meta Bans                : {total_meta_ban} Rules")
print(f"4. Aturan Combo Breaker Bans       : {total_combo_breaker} Rules")
print(f"5. Aturan Anti-Counter Bans        : {total_anti_counter} Rules")
print("==========================================")
print(f"TOTAL ATURAN TEREKSTRAKSI          : {total_keseluruhan} Rules")
print("==========================================")

     📊 RINGKASAN DATASET & ATURAN 📊     
Total Pertandingan : 558 Match
------------------------------------------
1. Aturan Basis Kawan (Allies)     : 79402 Rules
2. Aturan Basis Lawan (Counters)   : 3691 Rules
3. Aturan Meta Bans                : 19706 Rules
4. Aturan Combo Breaker Bans       : 16160 Rules
5. Aturan Anti-Counter Bans        : 16736 Rules
TOTAL ATURAN TEREKSTRAKSI          : 135695 Rules


In [9]:
# ==========================================
# Menyimpan seluruh aturan ke file CSV
# ==========================================
# Membuat DataFrame untuk setiap aturan
df_rules_allies = rules_allies[['antecedents', 'consequents', 'Team_Heroes', 'Rec_Ally', 'support', 'confidence', 'lift','win_rate']].copy()
df_rules_counters = rules_counters[['antecedents', 'consequents', 'Enemy', 'Rec_Counter', 'support', 'confidence', 'lift','counter_coef']].copy()
df_rules_meta_ban = rules_meta_ban[['antecedents', 'consequents', 'Target_Ban', 'support', 'confidence', 'lift']].copy()
df_rules_combo_breaker = rules_combo_breaker[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()
df_rules_anti_counter = rules_anti_counter[['antecedents', 'consequents', 'support', 'confidence', 'lift']].copy()

# ==========================================
# TRANSFORMASI FROZENSET KE STRING RAPI
# ==========================================
# Masukkan semua dataframe ke dalam list untuk di-loop
daftar_df_rules = [df_rules_allies, df_rules_counters, df_rules_meta_ban, df_rules_combo_breaker, df_rules_anti_counter]

for df in daftar_df_rules:
    # Ubah nilai menjadi string yang dipisahkan koma, untuk tipe data frozenset
    df['antecedents'] = df['antecedents'].apply(lambda x: ', '.join(list(x)) if isinstance(x, frozenset) else x)
    df['consequents'] = df['consequents'].apply(lambda x: ', '.join(list(x)) if isinstance(x, frozenset) else x)

# ==========================================
# Menyimpan DataFrame ke file CSV
# ==========================================
df_rules_allies.to_csv('../Extracted-ARM-Rules/rules_allies.csv', index=False)
df_rules_counters.to_csv('../Extracted-ARM-Rules/rules_counters.csv', index=False)
df_rules_meta_ban.to_csv('../Extracted-ARM-Rules/rules_meta_ban.csv', index=False)
df_rules_combo_breaker.to_csv('../Extracted-ARM-Rules/rules_combo_breaker.csv', index=False)
df_rules_anti_counter.to_csv('../Extracted-ARM-Rules/rules_anti_counter.csv', index=False)

print("Semua aturan ARM berhasil disimpan")

Semua aturan ARM berhasil disimpan
